<a href="https://colab.research.google.com/github/nivethithanm/python-systems-full/blob/main/DEEP_02_iterators_generators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DEEP-02: Iterators & Generators — Build the Protocol from Scratch

**Time**: ~2.5 hours | **Depth**: First principles

**Systems mapping**: `for x in seq` → `__iter__` → `__next__` → StopIteration → compiler sugar

You'll build: a lazy data pipeline (like a simplified PyTorch DataLoader).

In [1]:
import sys

## 1. The Iterator Protocol — What `for` Actually Does

In [2]:
# Exercise 1.1: Implement what 'for x in iterable: body' REALLY does

def manual_for(iterable, body_fn):
    """Replicate 'for x in iterable: body_fn(x)' without using for.

    Steps:
    1. Call iter(iterable) to get an iterator
    2. Repeatedly call next(iterator)
    3. Call body_fn(value) for each value
    4. Catch StopIteration to know when done
    """
    # YOUR CODE HERE
    iterator = iter(iterable)
    while iterator:
        try:
            value = next(iterator)
            body_fn((value))
        except:
            break

# Test: should produce same result as for loop
results = []
manual_for([10, 20, 30], lambda x: results.append(x * 2))
assert results == [20, 40, 60]
print('✅ 1.1 — you know what for-loops really do')

✅ 1.1 — you know what for-loops really do


In [3]:
# Exercise 1.2: Build a custom iterator class

class CountDown:
    """Iterator that counts down from n to 1.

    Implement __iter__ and __next__.
    __iter__ returns self (this IS the iterator).
    __next__ returns next value or raises StopIteration.
    """
    def __init__(self, n):
        # YOUR CODE HERE
        self.n = n

    def __iter__(self):
        # YOUR CODE HERE
        return self

    def __next__(self):
        # YOUR CODE HERE
        if self.n <= 0:
            raise StopIteration
        value = self.n
        self.n -= 1
        return value

assert list(CountDown(5)) == [5, 4, 3, 2, 1]
assert list(CountDown(0)) == []
print('✅ 1.2 — custom iterator')

✅ 1.2 — custom iterator


## 2. Generators — Pausable Functions

In [6]:
# Exercise 2.1: Implement the same CountDown as a generator function

def countdown_gen(n):
    """Generator that yields n, n-1, ..., 1."""
    # YOUR CODE HERE (use yield)
    i = n
    while i > 0:
        yield i
        i -= 1

assert list(countdown_gen(5)) == [5, 4, 3, 2, 1]

# Exercise 2.2: Understand generator state
# A generator PAUSES at yield and RESUMES on next()

def debug_gen():
    print('  START')
    yield 1
    print('  AFTER YIELD 1')
    yield 2
    print('  AFTER YIELD 2')
    # no more yields → StopIteration

g = debug_gen()
print('Created generator (nothing printed yet!)')
print(f'First next(): {next(g)}')
print(f'Second next(): {next(g)}')
try:
    next(g)
except StopIteration:
    print('StopIteration raised — generator exhausted')

print('✅ 2.2 — generator pause/resume')

Created generator (nothing printed yet!)
  START
First next(): 1
  AFTER YIELD 1
Second next(): 2
  AFTER YIELD 2
StopIteration raised — generator exhausted
✅ 2.2 — generator pause/resume


In [7]:
# Exercise 2.3: Memory comparison — list vs generator

# List: stores ALL values in memory
big_list = [i**2 for i in range(1_000_000)]
print(f'List memory: {sys.getsizeof(big_list):,} bytes')

# Generator: stores only the RECIPE, computes on demand
big_gen = (i**2 for i in range(1_000_000))
print(f'Generator memory: {sys.getsizeof(big_gen):,} bytes')

# YOUR TASK: explain WHY the generator uses ~100 bytes regardless of N
# Write your explanation as a comment:
# ANSWER: ...

List memory: 8,448,728 bytes
Generator memory: 200 bytes


## 3. Build a Lazy Pipeline (like PyTorch DataLoader)

In [16]:
# Exercise 3.1: Implement lazy map, filter, take, batch

class LazyPipeline:
    """Chainable lazy data pipeline.

    Usage:
        pipe = LazyPipeline(range(1000))
        result = pipe.filter(lambda x: x % 2 == 0).map(lambda x: x**2).take(5).collect()

    Key: NOTHING executes until .collect() is called.
    Each operation returns a new LazyPipeline wrapping a generator.
    """

    def __init__(self, iterable):
        self.source = iterable

    def map(self, fn) -> 'LazyPipeline':
        """Apply fn to each element lazily."""
        # YOUR CODE HERE
        # Hint: return LazyPipeline(fn(x) for x in self.source)
        return LazyPipeline(fn(x) for x in self.source)

    def filter(self, pred) -> 'LazyPipeline':
        """Keep elements where pred(x) is True, lazily."""
        # YOUR CODE HERE
        return LazyPipeline(x for x in self.source if pred(x))

    def take(self, n: int) -> 'LazyPipeline':
        """Take first n elements, lazily."""
        # YOUR CODE HERE
        # Hint: use a generator with a counter
        def gen():
            i = 0
            for x in self.source:
                if i == n:
                    break
                yield x
                i += 1
        return LazyPipeline(gen())

    def batch(self, size: int) -> 'LazyPipeline':
        """Group elements into batches of given size.
        Last batch may be smaller.
        """
        # YOUR CODE HERE
        def gen():
            count = 0
            batch = []
            for x in self.source:
                count += 1
                batch.append(x)
                if count == size:
                    yield batch
                    batch = []
                    count = 0
            if batch:
                yield batch
        return LazyPipeline(gen())

    def collect(self) -> list:
        """Materialize the pipeline into a list."""
        return list(self.source)

# Test
result = (LazyPipeline(range(100))
          .filter(lambda x: x % 3 == 0)   # 0, 3, 6, 9, ...
          .map(lambda x: x * 10)           # 0, 30, 60, 90, ...
          .take(5)                          # first 5
          .collect())
assert result == [0, 30, 60, 90, 120]

batches = (LazyPipeline(range(10))
           .batch(3)
           .collect())
assert batches == [[0,1,2], [3,4,5], [6,7,8], [9]]

print('✅ 3.1 — lazy pipeline (DataLoader pattern)')

✅ 3.1 — lazy pipeline (DataLoader pattern)


In [21]:
# Exercise 3.2: Build a mini DataLoader

import random

class MiniDataLoader:
    """Simplified PyTorch DataLoader using generators.

    Features:
    - Iterates over a dataset in batches
    - Optional shuffle at each epoch
    - Lazy — doesn't load all batches into memory
    - Supports multiple epochs via __iter__
    """

    def __init__(self, dataset: list, batch_size: int, shuffle: bool = False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        """Yield batches. Called once per epoch.
        If shuffle=True, shuffle indices (not the dataset itself).
        """
        # YOUR CODE HERE
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            random.shuffle(indices)
        for i in range(0, len(indices), self.batch_size):
            batch_indices = indices[i:i+self.batch_size]
            yield [self.dataset[i] for i in batch_indices]

    def __len__(self):
        """Number of batches per epoch."""
        # YOUR CODE HERE
        import math
        return math.ceil(len(self.dataset)/self.batch_size)

# Test
data = list(range(25))
loader = MiniDataLoader(data, batch_size=10, shuffle=False)

batches = list(loader)
assert len(batches) == 3  # 10 + 10 + 5
assert batches[0] == list(range(10))
assert batches[2] == [20, 21, 22, 23, 24]

# Two epochs should both work
epoch1 = list(loader)
epoch2 = list(loader)
assert len(epoch1) == len(epoch2) == 3

print('✅ 3.2 — mini DataLoader')

✅ 3.2 — mini DataLoader


## Mastery Check

- [ ] Explain what `for x in y` compiles to (iter + next + StopIteration)
- [ ] Build an iterator class AND a generator for the same sequence
- [ ] Explain why generators use O(1) memory regardless of sequence length
- [ ] Chain lazy operations without materializing intermediate results